In [5]:
# Cell 1: Install Dependencies
!pip install -q datasets torch transformers librosa yfinance pandas numpy

import os
from datasets import load_dataset
import pandas as pd
import librosa
import numpy as np

print("Downloading dataset from Hugging Face...")
try:
    # Load the correct 'test' split
    dataset = load_dataset("Revai/earnings21", split="test")
    sample_call = dataset[0]
    print(f"\nDataset loaded successfully! Total calls: {len(dataset)}")
    print(f"Sample Company Name: {sample_call['company_name']}")
    print(f"Curator ID: {sample_call['curator_id']}")
    print(f"Audio length: {sample_call['audio_length']} seconds")
    print(f"Transcript preview: {sample_call['text'][:200]}...")
except Exception as e:
    print(f"Direct load failed: {e}")
    print("Falling back to creating mock directory structure for MAEC/Earnings dataset to allow the code to run locally...")
    os.makedirs("corporate-risk-forecaster/data", exist_ok=True)
    with open("corporate-risk-forecaster/data/AAPL_2023-05-04_text.txt", "w") as f:
        f.write("We experienced significant headwinds this quarter, leading to a drop in EBITDA, though our NAV remains stable.")
    np.save("corporate-risk-forecaster/data/AAPL_2023-05-04_audio.npy", np.random.randn(500, 20))
    print("Mock environment configured with AAPL files under 'corporate-risk-forecaster/data'.")

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]


Dataset loaded successfully! Total calls: 44
Sample Company Name: Monro Inc
Curator ID: 1
Audio length: 3285.848 seconds
Transcript preview: Good morning ladies and gentlemen, and welcome to the Monro Inc Earnings Conference Call for the third quarter Fiscal 2020. At this time all participants are in listen only mode. Later, we will conduc...


In [48]:
# Cell 2: Ticker Mapper and Target Builder
import yfinance as yf
import pandas as pd
import numpy as np

# Expanded mapper to match companies inside the dataset, including Monro Inc
TICKER_MAP = {
    "Apple Inc.": "AAPL",
    "Alphabet Inc.": "GOOGL",
    "Microsoft Corporation": "MSFT",
    "Amazon.com, Inc.": "AMZN",
    "Monro Inc": "MNRO",
    "Monro, Inc.": "MNRO",
    "Adobe Inc.": "ADBE",
    "Walmart Inc.": "WMT",
    "eBay Inc.": "EBAY",
    # Add your mock fallback
    "Mock Company": "AAPL"
}

def calculate_forward_volatility(ticker, call_date="2021-04-22", window_days=7):
    """
    Fetches stock data and calculates the forward-looking log volatility.
    *Note: earnings21 calls are mostly from Spring 2021, so we default the date if missing.
    """
    try:
        start_date = pd.to_datetime(call_date) - pd.Timedelta(days=5)
        end_date = pd.to_datetime(call_date) + pd.Timedelta(days=window_days + 5)

        stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)

        if stock_data.empty: return None

        stock_data.index = stock_data.index.tz_localize(None)
        forward_data = stock_data[stock_data.index > pd.to_datetime(call_date)].head(window_days)

        if len(forward_data) < 2: return None

        log_returns = np.log(forward_data['Close'] / forward_data['Close'].shift(1)).dropna()
        return log_returns.std().item()

    except Exception as e:
        return None

# Test the function
print(f"Sample AAPL Volatility: {calculate_forward_volatility('AAPL')}")
print(f"Sample MNRO Volatility: {calculate_forward_volatility('MNRO')}")

Sample AAPL Volatility: 0.008022280774539326
Sample MNRO Volatility: 0.011835213048258192


/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


In [49]:
# Cell 3: Audio Signal Processing
import librosa
import torch

def extract_mfcc_features(raw_audio_array, sampling_rate, n_mfcc=20, max_frames=500):
    """
    Converts raw audio into a mathematical matrix of vocal features.
    """
    try:
        # Extract MFCCs using librosa
        # Output shape: (n_mfcc, time_frames)
        mfccs = librosa.feature.mfcc(y=raw_audio_array, sr=sampling_rate, n_mfcc=n_mfcc)

        # Transpose to (time_frames, n_mfcc) to match PyTorch LSTM expectations
        mfccs = mfccs.T

        # Truncate or pad to ensure a consistent length (e.g., 500 frames)
        if mfccs.shape[0] > max_frames:
            mfccs = mfccs[:max_frames, :]
        else:
            padding = max_frames - mfccs.shape[0]
            mfccs = np.pad(mfccs, ((0, padding), (0, 0)), mode='constant')

        return torch.tensor(mfccs, dtype=torch.float32)
    except Exception as e:
        print(f"Audio processing failed: {e}")
        # Return a zero tensor as a fallback
        return torch.zeros((max_frames, n_mfcc))

# If using the mock dataset, just load the numpy array you saved
try:
    dummy_audio = dataset[0]['audio']['array']
    dummy_sr = dataset[0]['audio']['sampling_rate']
    features = extract_mfcc_features(dummy_audio, dummy_sr)
    print(f"Extracted MFCC Shape from HF: {features.shape}")
except:
    features = torch.tensor(np.load("corporate-risk-forecaster/data/AAPL_2023-05-04_audio.npy"), dtype=torch.float32)
    print(f"Loaded Mock MFCC Shape: {features.shape}")

Extracted MFCC Shape from HF: torch.Size([500, 20])


In [50]:
# Cell 4: FinBERT Text Feature Extractor
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load ProsusAI's FinBERT
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
text_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
text_model.eval() # Set to evaluation mode

def get_text_features(text):
    """
    Passes transcript text through FinBERT and returns the raw logits
    (Positive, Negative, Neutral) to be used as features for the fusion model.
    """
    # Truncate to 512 tokens (BERT's maximum)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)

    with torch.no_grad():
        outputs = text_model(**inputs)

    # Return the raw logits [Positive, Negative, Neutral]
    return outputs.logits.cpu().numpy().flatten()

# Test the extractor
sample_text = "We experienced significant headwinds this quarter, leading to a drop in EBITDA."
text_features = get_text_features(sample_text)
print(f"FinBERT Features [Pos, Neg, Neu]: {text_features}")


Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT Features [Pos, Neg, Neu]: [-1.5797879  2.9615371 -1.1592702]


In [58]:
# Cell 5: Multimodal PyTorch Dataset and DataLoader
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch

class EarningsMultimodalDataset(Dataset):
    def __init__(self, hf_dataset, ticker_map):
        self.raw_data = hf_dataset
        self.ticker_map = ticker_map
        self.valid_samples = self._process_and_align()

    def _process_and_align(self):
        valid = []
        # Let's inspect the first 10 items in the dataset to find matches
        for i in range(( len(self.raw_data))):
            sample = self.raw_data[i]
            company = sample.get('company_name', "Mock Company")
            ticker = self.ticker_map.get(company)

            if not ticker:
                ticker = "AAPL"

            # 1. Get Target
            target = calculate_forward_volatility(ticker)
            if target is None:
                continue

            # 2. Get Text Features
            txt_feats = get_text_features(sample['text'][:2000]) # Truncating text for speed

            # 3. Get Audio Features
            try:
                audio_array = sample['audio']['array']
                sr = sample['audio']['sampling_rate']
            except Exception:
                # Fallback for mock data
                audio_array = np.random.randn(16000 * 10)
                sr = 16000

            audio_feats = extract_mfcc_features(audio_array, sr)

            valid.append({
                "text_features": torch.tensor(txt_feats, dtype=torch.float32),
                "audio_features": audio_feats,
                "label": torch.tensor([target], dtype=torch.float32)
            })
            print(f"Processed sample {i+1}: {company} -> {ticker} (Volatility: {target:.5f})")

        return valid

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, idx):
        sample = self.valid_samples[idx]
        return sample["text_features"], sample["audio_features"], sample["label"]

def collate_multimodal(batch):
    txt_list, aud_list, lbl_list = zip(*batch)

    # Text and labels are already uniform size
    txt_tensor = torch.stack(txt_list)
    lbl_tensor = torch.stack(lbl_list)

    # Audio length varies, so we pad with zeros to match the longest sequence in the batch
    aud_padded = pad_sequence(aud_list, batch_first=True, padding_value=0.0)

    return txt_tensor, aud_padded, lbl_tensor

# Initialize the Dataset and DataLoader with updated TICKER_MAP
multimodal_dataset = EarningsMultimodalDataset(dataset, TICKER_MAP)
print(f"Total matched and aligned dataset size: {len(multimodal_dataset)}")

if len(multimodal_dataset) > 0:
    train_loader = DataLoader(multimodal_dataset, batch_size=2, shuffle=True, collate_fn=collate_multimodal)

    # Fetch one batch to verify shapes
    for txt, aud, lbl in train_loader:
        print("\n--- Batch Shapes ---")
        print(f"Text Batch Shape: {txt.shape} -> (Batch Size, 3 FinBERT Logits)")
        print(f"Audio Batch Shape: {aud.shape} -> (Batch Size, Time Steps, 20 MFCCs)")
        print(f"Label Batch Shape: {lbl.shape} -> (Batch Size, 1 Volatility Target)")
        break
else:
    print("Error: Aligned dataset is empty. Check TICKER_MAP mappings or yfinance connectivity.")

/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


Processed sample 1: Monro Inc -> MNRO (Volatility: 0.01184)


/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


Processed sample 2: Culp Inc -> AAPL (Volatility: 0.00802)


/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


...

Processed sample 42: Innovative Solutions and Support Inc -> AAPL (Volatility: 0.00802)


/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


Processed sample 43: Rev Group Inc -> AAPL (Volatility: 0.00802)


/tmp/ipykernel_1847/2674195296.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)


Processed sample 44: Bed Bath and Beyond Inc -> AAPL (Volatility: 0.00802)
Total matched and aligned dataset size: 44

--- Batch Shapes ---
Text Batch Shape: torch.Size([2, 3]) -> (Batch Size, 3 FinBERT Logits)
Audio Batch Shape: torch.Size([2, 500, 20]) -> (Batch Size, Time Steps, 20 MFCCs)
Label Batch Shape: torch.Size([2, 1]) -> (Batch Size, 1 Volatility Target)


In [59]:
import torch.nn as nn
class MultimodalRiskForecaster(nn.Module):
  def __init__(self,audio_input_dim = 20, lstm_hidden = 64, text_dim = 3, output_dim= 1):
    super(MultimodalRiskForecaster, self).__init__()
    self.audio_lstm = nn.LSTM(
        input_size = audio_input_dim,
        hidden_size = lstm_hidden,
        num_layers = 2,
        batch_first=True,
        bidirectional = True
    )

    audio_feature_size = lstm_hidden * 2
    combined_dim = audio_feature_size + text_dim

    self.fc_layers = nn.Sequential(
        nn.Linear(combined_dim, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, output_dim),
        nn.Softplus()
    )

  def forward(self,text_features,audio_features):
    lstm_out, (hidden,cell) = self.audio_lstm(audio_features)
    forward_hidden  = hidden[-2,:,:]
    backward_hidden = hidden[-1,:,:]
    audio_vec = torch.cat((forward_hidden,backward_hidden),dim = 1)

    fused_vector = torch.cat((audio_vec,text_features),dim = 1)

    prediction = self.fc_layers(fused_vector)
    return prediction


fusion_model = MultimodalRiskForecaster().to(device)
print("Fusion Model initialized and moved to GPU")

Fusion Model initialized and moved to GPU


In [60]:
import torch
import torch.nn as nn

class AttentionRiskForecaster(nn.Module):
  def __init__(self, audio_input_dim=20, lstm_hidden=64, text_dim=3, embed_dim=64):
    super(AttentionRiskForecaster, self).__init__()

    self.audio_lstm = nn.LSTM(
        input_size=audio_input_dim,
        hidden_size=lstm_hidden,
        num_layers=2,
        batch_first=True,
        bidirectional=True
    )

    self.audio_proj = nn.Linear(lstm_hidden * 2, embed_dim)
    self.text_proj = nn.Linear(text_dim, embed_dim)

    self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=4, batch_first=True)

    self.fc_layers = nn.Sequential(
        nn.Linear(embed_dim, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Softplus()
    )

  def forward(self, text_features, audio_features):
    lstm_out, (hidden, cell) = self.audio_lstm(audio_features)

    audio_vec = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
    query = self.audio_proj(audio_vec).unsqueeze(1)
    key_value = self.text_proj(text_features).unsqueeze(1)

    attn_output, attn_weights = self.cross_attention(query, key_value, key_value)
    prediction = self.fc_layers(attn_output.squeeze(1))
    return prediction

In [61]:
from sklearn.metrics import root_mean_squared_error
import torch
import numpy as np

def evaluate_model(model, val_loader):
  model.eval()
  predictions, actuals = [], []

  with torch.no_grad():
    for txt, aud, lbl in val_loader:
      txt = txt.to(device)
      aud = aud.to(device)

      pred = model(txt, aud)
      predictions.extend(pred.cpu().numpy().flatten())
      actuals.extend(lbl.cpu().numpy().flatten())

  if len(actuals) == 0:
    return float('nan')

  rmse = root_mean_squared_error(actuals, predictions)
  return rmse

print("Evaluation function defined")

Evaluation function defined


In [62]:
# Cell 8: Walk-Forward Cross-Validation & Ablation
import torch
import torch.nn as nn
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, Subset

# Ensure dataset has enough samples for splits (adjust n_splits if sample count is small)
n_splits = min(3, len(multimodal_dataset) - 1) if len(multimodal_dataset) > 1 else 1

if len(multimodal_dataset) < 4:
    print(f"Dataset has {len(multimodal_dataset)} samples. Running a single train/val split instead of 5 folds.")
    train_size = max(1, int(0.7 * len(multimodal_dataset)))
    splits = [(list(range(train_size)), list(range(train_size, len(multimodal_dataset))))]
else:
    tscv = TimeSeriesSplit(n_splits=n_splits)
    splits = list(tscv.split(multimodal_dataset))

EPOCHS = 5
LEARNING_RATE = 0.001
criterion = nn.MSELoss()

attention_fold_rmse = []
baseline_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n=================== Fold {fold + 1} ===================")
    print(f"Train indices: {train_idx} | Val indices: {val_idx}")

    train_sub = Subset(multimodal_dataset, train_idx)
    val_sub = Subset(multimodal_dataset, val_idx)

    train_loader = DataLoader(train_sub, batch_size=2, shuffle=False, collate_fn=collate_multimodal)
    val_loader = DataLoader(val_sub, batch_size=2, shuffle=False, collate_fn=collate_multimodal)


    model_attn = AttentionRiskForecaster().to(device)
    optimizer_attn = torch.optim.Adam(model_attn.parameters(), lr=LEARNING_RATE)

    model_attn.train()
    for epoch in range(EPOCHS):
        for txt, aud, lbl in train_loader:
            txt, aud, lbl = txt.to(device), aud.to(device), lbl.to(device)
            optimizer_attn.zero_grad()
            preds = model_attn(txt, aud)
            loss = criterion(preds, lbl)
            loss.backward()
            optimizer_attn.step()

    rmse_attn = evaluate_model(model_attn, val_loader)
    attention_fold_rmse.append(rmse_attn)
    print(f"Fold {fold + 1} - Cross-Attention Model RMSE: {rmse_attn:.6f}")

    # ---------------------------------------------------------
    # 2. Train & Evaluate MultimodalRiskForecaster (Baseline)
    # ---------------------------------------------------------
    model_base = MultimodalRiskForecaster().to(device)
    optimizer_base = torch.optim.Adam(model_base.parameters(), lr=LEARNING_RATE)

    model_base.train()
    for epoch in range(EPOCHS):
        for txt, aud, lbl in train_loader:
            txt, aud, lbl = txt.to(device), aud.to(device), lbl.to(device)
            optimizer_base.zero_grad()
            preds = model_base(txt, aud)
            loss = criterion(preds, lbl)
            loss.backward()
            optimizer_base.step()

    rmse_base = evaluate_model(model_base, val_loader)
    baseline_fold_rmse.append(rmse_base)
    print(f"Fold {fold + 1} - Baseline Concatenation RMSE:  {rmse_base:.6f}")

print("\n=================== Ablation Summary ===================")
print(f"Mean Baseline RMSE:        {sum(baseline_fold_rmse) / len(baseline_fold_rmse):.6f}")
print(f"Mean Cross-Attention RMSE: {sum(attention_fold_rmse) / len(attention_fold_rmse):.6f}")


=================== Fold 1 ===================
Train indices: [ 0  1  2  3  4  5  6  7  8  9 10] | Val indices: [11 12 13 14 15 16 17 18 19 20 21]
Fold 1 - Cross-Attention Model RMSE: 0.230624
Fold 1 - Baseline Concatenation RMSE:  0.001632

=================== Fold 2 ===================
Train indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21] | Val indices: [22 23 24 25 26 27 28 29 30 31 32]
Fold 2 - Cross-Attention Model RMSE: 0.191306
Fold 2 - Baseline Concatenation RMSE:  0.006983

=================== Fold 3 ===================
Train indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32] | Val indices: [33 34 35 36 37 38 39 40 41 42 43]
Fold 3 - Cross-Attention Model RMSE: 0.026669
Fold 3 - Baseline Concatenation RMSE:  0.006118

=================== Ablation Summary ===================
Mean Baseline RMSE:        0.004911
Mean Cross-Attention RMSE: 0.149533


In [63]:
def predict_corporate_risk(text, raw_audio_array,sampling_rate):
  "takes raw transcripts and predicts raw volatility"

  fusion_model.eval()

  with torch.no_grad():
    text_feats = get_text_features(text)
    text_tensor = torch.tensor(text_feats,dtype = torch.float32).unsqueeze(0).to(device)

    audio_feats = extract_mfcc_features(raw_audio_array,sampling_rate)
    audio_tensor = audio_feats.unsqueeze(0).to(device)

    pred_volatility = fusion_model(text_tensor,audio_tensor)
    predicted_value = pred_volatility.item()


  risk_tier = "High" if predicted_value > 0.03 else("Medium" if predicted_value > 0.015 else "Low")

  return {
      "predicted_forward_volatility": predicted_value,
      "risk_tier": risk_tier,
      "text_sentiment_logits": text_feats.tolist()
  }

sample_audio = np.random.randn(16000*5)
sample_speech = "Operating cash flow declined sharply due to supply bottlenecks, impacting quarterly guidance."

result = predict_corporate_risk(sample_speech, sample_audio, 16000)
print("--- Inference Output ---")
print(f"Predicted Volatility: {result['predicted_forward_volatility']:.5f}")
print(f"Assessed Risk Tier:   {result['risk_tier']}")


--- Inference Output ---
Predicted Volatility: 0.65791
Assessed Risk Tier:   High


In [64]:
checkpoint_path = "multimodal_risk_forecaster.pt"
torch.save({
    'model_state_dict': fusion_model.state_dict(),
    'optimizer_state_dict': optimizer_attn.state_dict(),
    'model_config': {
        'audio_input_dim': 20,
        'lstm_hidden': 64,
        'text_dim': 3,
        'output_dim': 1
    }
}, checkpoint_path)

print(f"Model checkpoint successfully saved to {checkpoint_path}")

Model checkpoint successfully saved to multimodal_risk_forecaster.pt
